## Ecuaciones Hiperbólicas: Métodos Numéricos para Problemas Lineales y No Lineales

**Referencias:** LeVeque, *Finite Difference Methods for Ordinary and Partial Differential Equations*, Cap. 8-12; Morton & Mayers, *Numerical Solution of Partial Differential Equations*, Cap. 3-5.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurimendiluce/AN2026/blob/main/diferencias_finitas/clase_7_hiperbolicas2.ipynb)


### Teoría

Consideramos leyes de conservación escalares de primer orden
$$u_t + f(u)_x = 0,$$
con condición inicial $u(x,0)=u_0(x)$. Cuando $f(u)=au$ (caso **lineal**) la ecuación se resuelve exactamente por el método de las características: a lo largo de las rectas $x-at=\text{cte}$, $U$ es constante, de modo que $u(x,t)=u_0(x-at)$.

**Condición CFL.** Todo esquema explícito en una malla con paso espacial $\Delta x$ y temporal $\Delta t$ tiene un dominio de dependencia numérico acotado por la velocidad de propagación de la malla $\Delta x/\Delta t$. Para que este dominio contenga al dominio de dependencia analítico (la característica que pasa por $(x_j,t^{n+1})$) es **necesario** que
$$|\nu|=\left|\frac{a\Delta t}{\Delta x}\right|\le 1.$$
Esta es la condición de Courant-Friedrichs-Lewy (CFL); es necesaria para la convergencia de cualquier esquema explícito consistente, y en los esquemas de esta notebook también resulta **suficiente** para la estabilidad (von Neumann).

**Consistencia, estabilidad y convergencia.** Por el Teorema de Equivalencia de Lax (esquemas lineales, problemas bien puestos): *consistencia + estabilidad $\iff$ convergencia*. Para cada esquema analizamos:
1. el **error de truncado** (vía desarrollo de Taylor), que da el orden de consistencia;
2. el **factor de amplificación** $\lambda$ del análisis de von Neumann ($U_j^n=\lambda^n e^{ij\theta}$), que da la condición de estabilidad $|\lambda|\le 1\ \forall\theta$.

**Forma conservativa y el Teorema de Lax-Wendroff.** Cuando $f$ es no lineal (Parte II, ecuación de Burgers) las soluciones pueden desarrollar discontinuidades (choques) en tiempo finito aunque el dato inicial sea suave. La forma *conservativa* de la ecuación,
$$u_t+f(u)_x=0,$$
y la forma *no conservativa* (o semilineal, obtenida usando la regla de la cadena),
$$u_t+f'(u)u_x=0,$$
son equivalentes para soluciones clásicas ($C^1$), pero **no** para soluciones débiles con discontinuidades: solo la forma conservativa es consistente con la formulación integral de la ley de conservación, y por lo tanto solo un esquema en forma conservativa está garantizado (Teorema de Lax-Wendroff) a converger, si converge, a una solución débil que satisface la condición de salto. Esto se verifica numéricamente en la Parte II.

### Parte I — Ecuación de advección lineal

$$u_t+au_x=0,\qquad a=0.9>0$$

con dato inicial escalón
$$u_0(x)=\begin{cases}1 & 0.1<x<0.4\\ 0 & \text{en otro caso}\end{cases}$$

Por características, la solución exacta es $u(x,t)=u_0(x-at)$: el pulso se traslada rígidamente con velocidad $a$, sin cambiar de forma. Esta solución exacta es la que usamos como referencia para medir el error de cada esquema.

In [1]:
using Plots
using Printf

In [2]:
function u₀(x)
    if 0.1<x<=0.4
        return 1
    else
        return 0
    end
end

function u(x,t;a=1)
    return u₀(x-a*t)
end

u (generic function with 1 method)

### 1.1 Método Up-Wind

Con $\nu=\dfrac{\Delta t}{\Delta x}$ y $a>0$, discretizando $u_t$ por forward y $u_x$ por backward:
$$u_j^{n+1}=u_j^n-a\nu\left(u_j^n-u_{j-1}^n\right)$$

**Consistencia.** Desarrollando por Taylor alrededor de $(x_j,t^n)$ se obtiene el error de truncado
$$\tau_j^n=\frac{\Delta t}{2}u_{tt}+\frac{a\Delta x}{2}u_{xx}+O(\Delta t^2,\Delta x^2) = O(\Delta t,\Delta x),$$
es decir, el esquema es **consistente de orden 1**.

**Estabilidad (von Neumann).** Con $U_j^n=\lambda^ne^{ij\theta}$:
$$\lambda=1-a\nu(1-e^{-i\theta}) \;\Rightarrow\; |\lambda|^2 = 1-2a\nu(1-a\nu)(1-\cos\theta).$$
Se tiene $|\lambda|\le 1\ \forall\theta$ **si y solo si** $0\le a\nu\le 1$: la misma condición CFL, y además es suficiente para la estabilidad.

**Interpretación (difusión numérica).** Reemplazando $u_t=-au_x$ en el error de truncado se obtiene que Up-Wind resuelve, con error $O(\Delta x^2)$, la ecuación de advección-difusión
$$u_t+au_x=\underbrace{\frac{a\Delta x}{2}(1-a\nu)}_{\text{viscosidad numérica}\,\ge 0}U_{xx},$$
lo que explica el suavizado (amortiguamiento de las discontinuidades) observado en las simulaciones: el esquema es **disipativo**.

In [ ]:
function up_wind(u₀,u,ν,Δx,steps;a=1)

    x = 0:Δx:3
    U = zeros(length(x),steps)
    for j=1:length(x)
        U[j,1]=u₀(x[j])
    end

    Δt=ν*Δx 
    for n=2:steps
        for j=2:length(x)
            U[j,n]=U[j,n-1]-ν*a*(U[j,n-1]-U[j-1,n-1])
        end
    end

    U_ex=zeros(length(x),steps)
    for n=1:steps
        for j=1:length(x)
            U_ex[j,n]=u(x[j],Δt*n)
        end
    end
    return x,U,U_ex
end

In [ ]:
Δx = 0.02
ν=0.8
steps=500
x,U,U_ex = up_wind(u₀,u,ν,Δx,steps)
anim = @animate for n=1:steps
    plot(x,U_ex[:,n],ls=:dash,label="Solución exacta")
    plot!(x,U[:,n],label="Solución numérica (Up-Wind)")
end
gif(anim, "up_wind.gif", fps = 10)

**Comentario numérico.** Se observa el efecto disipativo predicho por la ecuación modificada: los bordes del pulso se suavizan progresivamente y su altura decae, pero **no** aparecen oscilaciones espurias — el esquema Up-Wind satisface un principio del máximo discreto (los coeficientes $a\nu$ y $1-a\nu$ son ambos $\ge 0$ y suman 1 bajo CFL, por lo que $U_j^{n+1}$ es un promedio convexo de $U_j^n,U_{j-1}^n$).

### 1.2 Método Lax-Wendroff

Se obtiene por el "procedimiento de Lax-Wendroff": desarrollo de Taylor en tiempo
$$u(x,t+\Delta t)=u+\Delta t\,u_t+\frac{\Delta t^2}{2}u_{tt}+O(\Delta t^3),$$
y usando la ecuación para reemplazar derivadas temporales por espaciales, $u_t=-au_x \Rightarrow u_{tt}=a^2u_{xx}$, discretizadas centradas:
$$U_j^{n+1}=U_j^n-\frac{a\nu}{2}\left(U_{j+1}^n-U_{j-1}^n\right)+\frac{a^2\nu^2}{2}\left(U_{j+1}^n-2U_j^n+U_{j-1}^n\right)$$

**Consistencia.** Por construcción, los términos $O(\Delta t)$ y $O(\Delta t^2)$ del desarrollo temporal quedan absorbidos, y el error de truncado es $\tau=O(\Delta t^2,\Delta x^2)$: es **consistente de orden 2**, a diferencia de Up-Wind.

**Estabilidad.** El factor de amplificación es
$$\lambda=1-ia\nu\sin\theta-a^2\nu^2(1-\cos\theta),\qquad |\lambda|^2=1-a^2\nu^2(1-a^2\nu^2)(1-\cos\theta)^2,$$
y $|\lambda|\le1\ \forall\theta \iff |a\nu|\le 1$: la misma condición CFL vuelve a ser suficiente.

**Naturaleza dispersiva (sin principio del máximo).** A diferencia de Up-Wind, la ecuación modificada de Lax-Wendroff no tiene un término disipativo: su primer término de error es dispersivo ($\propto U_{xxx}$), no difusivo. En la representación
$$U_j^{n+1}=\tfrac12\nu(1+\nu)U_{j-1}^n+(1-\nu^2)U_j^n-\tfrac12\nu(1-\nu)U_{j+1}^n,$$
con $\nu=a\nu\in(0,1]$ el coeficiente de $U_{j+1}^n$ es **negativo**: $U_j^{n+1}$ es una combinación de tres valores con un peso negativo, por lo que **no** hay principio del máximo discreto. Esto explica las oscilaciones tipo Gibbs que se observan cerca de discontinuidades, ausentes cuando el dato inicial es suave (ver más abajo).

In [ ]:
function lax_wendroff(u₀,u,ν,Δx,steps;a=1)

    x=0:Δx:3
    U = zeros(length(x),steps)
    for j=1:length(x)
        U[j,1]=u₀(x[j])
    end

    Δt=ν*Δx
    for n=2:steps
        for j=2:length(x)-1
            U[j,n]=U[j,n-1]-0.5*a*ν*(U[j+1,n-1]-U[j-1,n-1])+0.5*(a*ν)^2*(U[j+1,n-1]-2*U[j,n-1]+U[j-1,n-1])
        end
    end

    U_ex=zeros(length(x),steps)
    for n=1:steps
        for j=1:length(x)
            U_ex[j,n]=u(x[j],Δt*n)
        end
    end

    return x,U,U_ex
end

In [ ]:
Δx = 0.02
steps=100
ν = 0.7
x,U,U_ex = lax_wendroff(u₀,u,ν,Δx,steps)
anim = @animate for n=1:steps
    plot(x,U_ex[:,n],ls=:dash,label="Solución exacta")
    plot!(x,U[:,n],label="Solución numérica (Lax-Wendroff)")
end
gif(anim, "lax_wendroff.gif", fps = 10)

**Comentario:** Las oscilaciones surgen porque el esquema de Lax-Wendroff no satisface un principio de máximo, como se discutió arriba: con $\nu=a\Delta t/\Delta x\in(0,1]$ el coeficiente de $U_{j+1}^n$ en
$$U^{n+1}_j = \frac{1}{2}\nu(1 + \nu)U^n_{j-1} + (1 - \nu^2)U^n_j - \frac{1}{2} \nu(1 - \nu)U^n_{j+1}$$
es negativo. Por lo tanto $U^{n+1}_j$ es un promedio ponderado con un peso negativo, y la solución numérica puede desarrollar oscilaciones con máximos y mínimos internos.

**Caso suave.** Repetimos el análisis con un dato inicial suave (gaussiana) en lugar del escalón discontinuo:
$$u(x,0)=e^{-10(4x-1)^2}.$$
Al ser $u_0\in C^\infty$, el error de truncado $O(\Delta x^2)$ realmente domina el comportamiento del esquema, y con la misma malla que antes el error es considerablemente menor: sigue habiendo un rastro de oscilación dispersiva a la izquierda del pulso, pero mucho más pequeño que en el caso discontinuo, y se reduce visiblemente al refinar la malla.

In [ ]:
u₀₀(x)=exp(-10*(4x-1)^2)

uu(x,t;a=1) = u₀₀(x-a*t)

In [ ]:
Δx = 0.01
steps=200
x,U,U_ex = lax_wendroff(u₀₀,uu,ν,Δx,steps)
anim = @animate for n=1:steps
    plot(x,U_ex[:,n],ls=:dash,label="Solución exacta")
    plot!(x,U[:,n],label="Solución numérica (Lax-Wendroff)")
end
gif(anim, "lax_wendroff_suave.gif", fps = 10)

### Lax-Friedrichs

Parte de la discretización centrada e **inestable** $u_j^{n+1}=u_j^n-\tfrac12 a\nu(u_{j+1}^n-u_{j-1}^n)$ (que corresponde a $g(\theta)=1-ia\nu\sin\theta$, con $|g(\theta)|>1$ para todo $\theta\ne 0,\pi$: siempre inestable) y la estabiliza reemplazando $u_j^n$ por el promedio de sus vecinos:
$$u_j^{n+1}=\frac12\left(u_{j+1}^n+u_{j-1}^n\right)-\frac{a\nu}{2}\left(u_{j+1}^n-u_{j-1}^n\right)$$

**Consistencia.** Escribiendo $\tfrac12(u_{j+1}^n+u_{j-1}^n)=u_j^n+\tfrac{\Delta x^2}{2}U_{xx}+O(\Delta x^4)$, el término agregado introduce un error $O(\Delta x^2/\Delta t)$; si $\Delta t,\Delta x\to0$ con $\nu$ fijo, este término es $O(\Delta x)$, por lo que el esquema es **consistente de orden 1** (igual que Up-Wind), y consistente solo si $\Delta x^2/\Delta t\to 0$.

**Estabilidad.** $g(\theta)=\cos\theta-ia\nu\sin\theta \Rightarrow |g(\theta)|^2=\cos^2\theta+a^2\nu^2\sin^2\theta \le 1 \iff |a\nu|\le1$: nuevamente la condición CFL.

**Difusión numérica.** El promedio $\tfrac12(u_{j+1}^n+u_{j-1}^n)=u_j^n+\tfrac{\Delta x^2}{2}U_{xx}+\cdots$ agrega, comparado con la discretización centrada inestable, un término de la forma $\tfrac{\Delta x^2}{2\Delta t}U_{xx}=\tfrac{a\Delta x}{2\nu}U_{xx}$: es un esquema **disipativo**, y de hecho más difusivo que Up-Wind para un mismo $\nu<1$ (viscosidad numérica $\propto 1/\nu$ en lugar de $\propto(1-a\nu)$), lo cual se aprecia comparando ambas simulaciones.

In [ ]:
function lax_friedrichs(u₀,u,ν,Δx,steps;a=1)

    x = 0:Δx:3
    U = zeros(length(x),steps)
    for j=1:length(x)
        U[j,1]=u₀(x[j])
    end

    Δt=ν*Δx
    for n=2:steps
        for j=2:length(x)-1
            U[j,n]=0.5*(U[j+1,n-1]+U[j-1,n-1])-0.5*a*ν*(U[j+1,n-1]-U[j-1,n-1])
        end
    end

    U_ex=zeros(length(x),steps)
    for n=1:steps
        for j=1:length(x)
            U_ex[j,n]=u(x[j],Δt*n)
        end
    end
    return x,U,U_ex
end

In [ ]:
Δx = 0.02
ν=0.8
steps=500
x,U,U_ex = lax_friedrichs(u₀,u,ν,Δx,steps)
anim = @animate for n=1:steps
    plot(x,U_ex[:,n],ls=:dash,label="Solución exacta")
    plot!(x,U[:,n],label="Solución numérica (Lax-Friedrichs)")
end
gif(anim, "lax_friedrichs.gif", fps = 10)

### Orden de convergencia

Confirmamos los órdenes teóricos (Up-Wind, Lax-Friedrichs, Lax-Wendroff) midiendo el error en norma $L^2$ discreta contra la solución exacta, con el dato inicial **suave** (gaussiano, para que la solución sea $C^\infty$ y el orden asintótico sea visible sin el efecto de la discontinuidad), a número de Courant $\nu=a\Delta t/\Delta x$ fijo, refinando $\Delta x$. El orden se estima con la fórmula local
$$\alpha\approx\frac{\log(E_i/E_{i+1})}{\log(h_i/h_{i+1})}$$
entre pasos sucesivos de refinamiento (no con un ajuste global), para poder ver el comportamiento pre-asintótico.

In [10]:
function error_L2(u_num, u_ex, Δx)
    return sqrt(Δx*sum((u_num.-u_ex).^2))
end

function tabla_convergencia(esquema, u₀, u; a=1, cfl=0.8, Tfinal=0.5,
                             Δxs=[0.02,0.01,0.005,0.0025])
    errores = Float64[]
    @printf("%-10s %-14s %-10s\n", "Δx", "error L2", "orden α")
    for Δx in Δxs
        Δt = cfl*Δx/a
        steps = round(Int, Tfinal/Δt)
        x,U,U_ex = esquema(u₀₀,uu,cfl,Δx,steps;a=a)
        e = error_L2(U[:,steps], U_ex[:,steps], Δx)
        push!(errores, e)
        @printf("%-10.4f %-14.6e\n", Δx, e)
    end
    for i in 1:length(Δxs)-1
        α = log(errores[i]/errores[i+1])/log(Δxs[i]/Δxs[i+1])
        @printf("  orden Δx=%.4f -> %.4f :  α ≈ %.3f\n", Δxs[i], Δxs[i+1], α)
    end
    return Δxs, errores
end

tabla_convergencia (generic function with 1 method)

In [ ]:
println("Up-Wind:")
Δxs, e_upwind = tabla_convergencia(up_wind, u₀₀, uu)
println()
println("Lax-Wendroff:")
_, e_lw = tabla_convergencia(lax_wendroff, u₀₀, uu)
println()
println("Lax-Friedrichs:")
_, e_lf = tabla_convergencia(lax_friedrichs, u₀₀, uu)

In [ ]:
plot(Δxs, e_upwind, xscale=:log10, yscale=:log10, marker=:circle, label="Up-Wind")
plot!(Δxs, e_lw, marker=:square, label="Lax-Wendroff")
plot!(Δxs, e_lf, marker=:diamond, label="Lax-Friedrichs")
# rectas de referencia de pendiente 1 y 2
plot!(Δxs, e_upwind[1].*(Δxs./Δxs[1]), ls=:dash, lc=:gray, label="pendiente 1")
plot!(Δxs, e_lw[1].*(Δxs./Δxs[1]).^2, ls=:dot, lc=:gray, label="pendiente 2")
xlabel!("Δx"); ylabel!("error L2"); title!("Convergencia: escala log-log")

### Parte II — Ecuación de Burgers: esquemas conservativos y no conservativos

$$u_t+\tfrac12(u^2)_x=0 \qquad \text{(forma conservativa)}$$
$$u_t+uu_x=0 \qquad \text{(forma no conservativa / semilineal)}$$

con el mismo dato inicial escalón que en la Parte I:
$$u_0(x)=\begin{cases}1 & 0.1<x<0.4\\ 0 & \text{en otro caso}\end{cases}$$

A diferencia de la Parte I, aquí $f(u)=u^2/2$ es **no lineal**: la velocidad característica es $f'(u)=u$, distinta en cada punto según el valor de $u$. Esto tiene dos consecuencias fundamentales que no aparecen en el caso lineal.

**Formación de choques.** Las características son rectas $x=x_0+u_0(x_0)t$. En $x_0=0.4^-$ (borde derecho del escalón) las características con velocidad $1$ (dentro del pulso) alcanzan a las de velocidad $0$ (fuera), y se cruzan en tiempo finito: se forma un **choque** (discontinuidad) en $x=0.4$. En cambio en $x_0=0.1$ (borde izquierdo) la velocidad pasa de $0$ a $1$ hacia la derecha.

**Condición de Rankine-Hugoniot.** Cuando hay una discontinuidad, la única noción de solución razonable es la solución débil (integral). Integrando la ley de conservación en una caja que cruza el choque se obtiene la velocidad del choque
$$s=\frac{f(u_L)-f(u_R)}{u_L-u_R},$$
donde $u_L,u_R$ son los valores a izquierda y derecha del choque. Para nuestro choque en $x=0.4$: $U_L=1$, $U_R=0$, así que
$$s=\frac{\tfrac12(1)^2-\tfrac12(0)^2}{1-0}=\frac12=0.5.$$
Es decir, la posición del choque debería evolucionar como $x_s(t)=0.4+0.5\,t$.

**¿Por qué importa la forma conservativa?** La forma no conservativa $u_t+uu_x=0$ se obtiene de la conservativa usando la regla de la cadena, válida **solo si $u$ es diferenciable**. En presencia de un choque esta manipulación no está justificada: la forma no conservativa ya no es equivalente a la ley de conservación original. El **Teorema de Lax-Wendroff** garantiza que, si una sucesión de soluciones de un esquema en forma *conservativa* converge, el límite es una solución débil de la ley de conservación. Un esquema en forma no conservativa no tiene esa garantía: puede converger a una función que **no** es una solución débil válida, con velocidad de choque incorrecta. Esto se verifica numéricamente a continuación.

In [17]:
using Plots
using Printf

In [ ]:
function u₀(x)
    if 0.1<x<=0.4
        return 1
    else
        return 0
    end
end

function u(x,t;a=0.9)
    return u₀(x-a*t)
end

### 2.1 Esquema conservativo (Up-Wind sobre el flujo)

Discretizando el flujo $f(u)=u^2/2$ con Up-Wind (backward, ya que $u\ge0$ aquí implica $f'(u)=U\ge0$):
$$U_j^{n+1}=U_j^n-\frac{\nu}{2}\left[(U_j^n)^2-(U_{j-1}^n)^2\right]$$
Esta discretización es de la forma conservativa $U_j^{n+1}=U_j^n-\nu(\hat f_{j+1/2}-\hat f_{j-1/2})$ con flujo numérico $\hat f_{j-1/2}=\tfrac12(U_{j-1}^n)^2$, y por lo tanto hereda las propiedades de conservación discreta (telescopía de la suma $\sum_j U_j^{n+1}\Delta x$ salvo términos de borde) requeridas por el Teorema de Lax-Wendroff.

### 2.2 Esquema no conservativo

Discretizando directamente la forma semilineal $u_t+uu_x=0$ con Up-Wind:
$$U_j^{n+1}=U_j^n-\nu\, U_j^n\left(U_j^n-U_{j-1}^n\right)$$
Para soluciones suaves esto es consistente con la misma ecuación que el esquema conservativo (ambas formas coinciden si $u\in C^1$), pero **no** puede escribirse como una diferencia de flujos discretos: no es un esquema conservativo, y no aproxima necesariamente la solución débil correcta cuando aparece un choque.

In [ ]:
function burgers_conservativo(u₀,ν,Δx,steps)
    x=0:Δx:2
    U = zeros(length(x),steps)
    for j=1:length(x)
        U[j,1] = u₀(x[j])
    end
    Δt = ν*Δx
    for n=2:steps
        for j=2:length(x)
            U[j,n]=U[j,n-1]-ν*0.5*(U[j,n-1]^2-U[j-1,n-1]^2)
        end
    end
    return U
end


function burgers_no_conservativo(u₀,ν,Δx,steps)
    x=0:Δx:2
    U = zeros(length(x),steps)
    for j=1:length(x)
        U[j,1] = u₀(x[j])
    end
    Δt = ν*Δx
    for n=2:steps
        for j=2:length(x)
            U[j,n]=U[j,n-1]-ν*U[j,n-1]*(U[j,n-1]-U[j-1,n-1])
        end
    end
    return U
end

In [ ]:
steps = 100
ν = 0.8

u_cons=burgers_conservativo(u₀,ν,0.01,100)
u_nocons=burgers_no_conservativo(u₀,ν,0.01,100)

x=0:0.01:2
plot(x,u_cons[:,steps],label="Conservativo")
plot!(x,u_nocons[:,steps],label="No Conservativo")

In [ ]:
anim = @animate for n=1:steps
    plot(x,u_cons[:,n],ls=:dash,label="Conservativo")
    plot!(x,u_nocons[:,n],label="No Conservativo")
end
gif(anim, "conservativo.gif", fps = 10)